# 🏭 Warehouse & Master Data — Complete Reference (`ws_mch_iot`)

**Storage and Dims folder.** Source-of-truth master data (plants, lines, machines) with **SCD Type-2**
history, a current-state **view**, and **RLS + CLS** security. This is the dimensional reference the
real-time telemetry (`tbl_mch_clean.machine_id`) joins to.

| Item | Type | Role |
|---|---|---|
| `wh_mch` | Warehouse | dim tables, SCD2 proc, view, RLS/CLS |
| `lh_mch` | Lakehouse | OneLake landing / notebook I/O |

Built headless on the **Python kernel** (`pyodbc` + ODBC Driver 18 + Entra token) — no SQL/Spark kernel.
Roster generated from shared `roster.py` so ids match the live telemetry exactly.


## 1. Dimension tables (`wh_mch.dim` schema)

### `DIM_PLANT` (10 rows)
`plant_sk, plant_id, plant_name, country, region, city, timezone, plant_manager, commissioned_date,
is_active, scd_start_date, scd_end_date, scd_is_current`

### `DIM_LINE` (18 rows)
`line_sk, line_id, plant_id(FK), line_name, product_category, max_throughput_per_hr, supervisor,
commissioned_date, is_active, scd_start_date, scd_end_date, scd_is_current`

### `DIM_MACHINE` (31 current + SCD history rows) — full SCD Type-2
`machine_sk, machine_id, line_id(FK), machine_name, manufacturer, model, machine_type, install_date,
rated_power_kw, max_rpm, max_temp_c, max_vibration_mm_s, criticality, firmware_version,
maintenance_cost*, vendor_contact*, last_maintenance_date, is_active, scd_start_date, scd_end_date,
scd_is_current`   (* = CLS-protected, see §4)

**Roster:** 10 plants · 18 lines · 31 machines, full referential integrity (every machine→line→plant
resolves). Original ids preserved: `MACH_1001/1002/2001/2002`.


## 2. SCD Type-2 — history tracking

`scd_start_date` / `scd_end_date` / `scd_is_current` bracket each version. The current row has
`scd_end_date = NULL, scd_is_current = 1`.

### Proc `dim.sp_mch_scd2_machine(@machine_id, @firmware, @maint_cost, @last_maint)`
1. If a tracked attribute changed vs the current row →
2. **expire** the current row (`scd_end_date = now, scd_is_current = 0`), then
3. **insert** a new current version carrying static attributes forward + the new values.

### History present in this build
| Machine | Versions |
|---|---|
| `MACH_1001` | 3 (v1.0 → v2.0 → v2.1) |
| `MACH_2001`, `MACH_2002`, `MACH_1002`, `MACH_1003` | 2 each |

Query current only: `WHERE scd_is_current = 1`. Point-in-time: `WHERE @t BETWEEN scd_start_date AND
ISNULL(scd_end_date,'9999-12-31')`.

## 3. Serving view — `dim.vw_mch_machine_current`
Current machines (`scd_is_current=1`) joined to their current line + plant — the clean "what is the
fleet right now" surface for BI, hiding SCD plumbing. 31 rows.


## 4. Security (minimal, demo-level)

### Row-Level Security (RLS) — `dim.rls_plant`
Predicate `dim.fn_rls_region(region)`: members of role **`mch_operator`** see only **APAC** plants;
everyone else sees all. Policy state = ON.

### Column-Level Security (CLS)
`DENY SELECT` on `DIM_MACHINE(maintenance_cost)` and `DIM_MACHINE(vendor_contact)` to **`mch_operator`** —
operators never see commercial/vendor columns.

> Note: Fabric Warehouse has **no `DATABASE_PRINCIPAL_ID()`** — role checks use `sys.database_principals`.
> `DATETIME2` requires explicit precision → `DATETIME2(6)`.

## 5. Lineage
```
tbl_mch_clean.machine_id ──▶ DIM_MACHINE.machine_id (scd_is_current=1)
DIM_MACHINE.line_id      ──▶ DIM_LINE.line_id
DIM_LINE.plant_id        ──▶ DIM_PLANT.plant_id
vw_mch_machine_current   = current join of all three
```

## 6. Operate / rebuild
- `nb_mch_dim_scd2` (Build and Setup) — rebuilds dims from `roster.py`, applies SCD2 changes, creates the view.
  It drops `rls_plant`/`fn_rls_region` first (FK on DIM_PLANT), so **re-run `nb_mch_security` after** to re-apply RLS+CLS.
- `nb_mch_security` — applies RLS + CLS (idempotent).

See the Real-Time Intelligence doc for the telemetry/eventhouse side.
